# Training Data Quality Analysis

The training set contains ~4000 echogram patches paired with bottom-detection masks.
Some patches are too noisy to interpret, have no discernible bottom, or have masks
generated from failed bottom detection. This notebook computes per-patch quality
metrics, ranks patches from best to worst, and provides a visual browser with
copy-pasteable identifiers so bad training pairs can be flagged and removed.

The approach: compute a battery of signal, mask, and boundary metrics on every patch
in a single memory-efficient pass (patches stay on disk, only the metrics DataFrame
lives in RAM). Then visualise the extremes to see whether the metrics actually
separate good from bad data.

In [ ]:
import sys
sys.path.insert(0, '../..')
from project_paths import REPO_ROOT

import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

PATCH_DIR = REPO_ROOT / 'Datasets/patch_cache'
OUTPUT_CSV = REPO_ROOT / 'Datasets/patch_quality_metrics.csv'
FLAGGED_JSON = REPO_ROOT / 'Datasets/flagged_bad_patches.json'

npz_files = sorted(PATCH_DIR.glob('*.npz'))
print(f'Found {len(npz_files)} npz files in {PATCH_DIR}')

## Metric functions

Each metric targets a specific failure mode:

- **zero_frac** — fraction of pixels that are exactly 0.0 in channel 0. During normalisation NaN values are filled with 0.0, so a high zero fraction signals missing data.
- **dynamic_range** — 95th minus 5th percentile of channel 0. Low dynamic range means the patch is featureless noise.
- **water_col_mean / water_col_std** — mean and standard deviation of channel 0 above the detected bottom. Clean water should be dark and uniform; high values indicate noise or scattering layers.
- **bottom_coverage** — fraction of pings (columns) where a bottom was actually detected. Low coverage means the mask was mostly interpolated.
- **bottom_smoothness** — standard deviation of the per-ping bottom depth index. A jagged or erratic bottom line suggests detection failure.
- **bottom_frac** — fraction of the mask that is labelled as bottom. Extreme values (very high or very low) are suspicious.
- **bottom_depth_range** — max minus min of the bottom depth index across pings. Large range within a 256-ping window suggests a steep slope or detection artefact.
- **boundary_contrast** — mean absolute intensity difference between a 5-pixel band above and below the detected bottom. A real seabed return should produce a sharp contrast; low values suggest the mask boundary doesn't correspond to a physical feature.
- **freq_coherence** — Pearson correlation between channel 0 (38 kHz) and channel 1 (200 kHz) in a band around the bottom. Both frequencies should show the bottom at the same depth.

In [ ]:
def get_bottom_line(mask):
    """Extract per-ping bottom depth index from a binary mask.

    For each column, finds the first row where mask == 1 (the top of the
    'below-bottom' region). Returns -1 for columns with no bottom detected.

    Parameters
    ----------
    mask : ndarray, shape (H, W), uint8

    Returns
    -------
    bottom : ndarray, shape (W,), int
    """
    H, W = mask.shape
    bottom = np.full(W, -1, dtype=int)
    for j in range(W):
        col = mask[:, j]
        if col.any():
            bottom[j] = np.argmax(col)
    return bottom


def compute_patch_metrics(patch, mask):
    """Compute all quality metrics for a single patch + mask pair.

    Parameters
    ----------
    patch : ndarray, shape (3, H, W), float32
    mask  : ndarray, shape (H, W), uint8

    Returns
    -------
    metrics : dict
    """
    ch0 = patch[0]  # 38 kHz, [0, 1]
    ch1 = patch[1]  # 200 kHz, [0, 1]
    H, W = ch0.shape

    bottom = get_bottom_line(mask)
    valid = bottom >= 0
    n_valid = valid.sum()

    m = {}

    # --- Signal quality ---
    m['zero_frac'] = float((ch0 == 0.0).mean())
    m['dynamic_range'] = float(np.percentile(ch0, 95) - np.percentile(ch0, 5))

    # Water column stats (pixels above the bottom line)
    if n_valid > 0:
        water_pixels = []
        for j in np.where(valid)[0]:
            if bottom[j] > 5:  # need at least a few rows of water
                water_pixels.append(ch0[:bottom[j], j])
        if water_pixels:
            water = np.concatenate(water_pixels)
            m['water_col_mean'] = float(water.mean())
            m['water_col_std'] = float(water.std())
        else:
            m['water_col_mean'] = np.nan
            m['water_col_std'] = np.nan
    else:
        m['water_col_mean'] = np.nan
        m['water_col_std'] = np.nan

    # --- Mask / bottom quality ---
    m['bottom_coverage'] = float(n_valid / W)
    m['bottom_frac'] = float(mask.mean())

    if n_valid > 1:
        valid_depths = bottom[valid]
        m['bottom_smoothness'] = float(valid_depths.std())
        m['bottom_depth_range'] = float(valid_depths.max() - valid_depths.min())
        # Max single-ping jump
        diffs = np.abs(np.diff(valid_depths))
        m['bottom_gradient_max'] = float(diffs.max()) if len(diffs) > 0 else 0.0
    else:
        m['bottom_smoothness'] = np.nan
        m['bottom_depth_range'] = np.nan
        m['bottom_gradient_max'] = np.nan

    # --- Boundary quality ---
    if n_valid > 0:
        band = 5
        contrasts = []
        coherence_above_0, coherence_above_1 = [], []

        for j in np.where(valid)[0]:
            b = bottom[j]
            above_start = max(0, b - band)
            below_end = min(H, b + band)
            if b - above_start < 2 or below_end - b < 2:
                continue
            above = ch0[above_start:b, j].mean()
            below = ch0[b:below_end, j].mean()
            contrasts.append(abs(below - above))

            # For frequency coherence — collect values in a band around bottom
            coherence_above_0.append(ch0[above_start:below_end, j])
            coherence_above_1.append(ch1[above_start:below_end, j])

        m['boundary_contrast'] = float(np.mean(contrasts)) if contrasts else np.nan

        # Frequency coherence at the boundary region
        if len(coherence_above_0) > 10:
            vec0 = np.concatenate(coherence_above_0)
            vec1 = np.concatenate(coherence_above_1)
            if vec0.std() > 1e-6 and vec1.std() > 1e-6:
                m['freq_coherence'] = float(np.corrcoef(vec0, vec1)[0, 1])
            else:
                m['freq_coherence'] = np.nan
        else:
            m['freq_coherence'] = np.nan
    else:
        m['boundary_contrast'] = np.nan
        m['freq_coherence'] = np.nan

    return m


# Quick sanity check on one patch
d = np.load(npz_files[0])
test_metrics = compute_patch_metrics(d['patches'][0], d['masks'][0])
for k, v in test_metrics.items():
    print(f'  {k:>22s}: {v:.4f}' if not np.isnan(v) else f'  {k:>22s}: NaN')

## Compute metrics for all patches

Single pass over every npz file. Each patch gets a uid of the form
`filename_stem::patch_index` that can be used to reload it later.

In [ ]:
records = []

for npz_path in tqdm(npz_files, desc='Computing metrics'):
    stem = npz_path.stem
    source_zarr = '_'.join(stem.split('_')[:-1])  # drop the hash suffix
    data = np.load(npz_path)
    patches, masks = data['patches'], data['masks']
    n = patches.shape[0]

    for i in range(n):
        uid = f'{stem}::{i}'
        m = compute_patch_metrics(patches[i], masks[i])
        m['uid'] = uid
        m['npz_file'] = stem
        m['patch_idx'] = i
        m['source_zarr'] = source_zarr
        m['height'] = patches.shape[2]
        records.append(m)

    del data, patches, masks  # free memory between files

df = pd.DataFrame(records)
del records

# Reorder columns so identifiers come first
id_cols = ['uid', 'npz_file', 'patch_idx', 'source_zarr', 'height']
metric_cols = [c for c in df.columns if c not in id_cols]
df = df[id_cols + metric_cols]

df.to_csv(OUTPUT_CSV, index=False)
print(f'Computed metrics for {len(df)} patches across {df["npz_file"].nunique()} files')
print(f'Saved to {OUTPUT_CSV}')
df.head()

## Metric distributions

In [ ]:
METRIC_COLS = [
    'zero_frac', 'dynamic_range', 'water_col_mean', 'water_col_std',
    'bottom_coverage', 'bottom_smoothness', 'bottom_frac',
    'bottom_depth_range', 'bottom_gradient_max',
    'boundary_contrast', 'freq_coherence',
]

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flat

for i, col in enumerate(METRIC_COLS):
    ax = axes[i]
    vals = df[col].dropna()
    ax.hist(vals, bins=50, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(col, fontsize=10)
    ax.set_ylabel('count')
    p5, p50, p95 = np.percentile(vals, [5, 50, 95])
    ax.axvline(p5, color='orange', ls='--', lw=1, label=f'p5={p5:.3f}')
    ax.axvline(p50, color='red', ls='-', lw=1, label=f'p50={p50:.3f}')
    ax.axvline(p95, color='orange', ls='--', lw=1, label=f'p95={p95:.3f}')
    ax.legend(fontsize=7, loc='upper right')

# Hide unused subplot
for j in range(len(METRIC_COLS), len(list(axes))):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

print(df[METRIC_COLS].describe().round(4).to_string())

## Metric correlations

In [ ]:
corr = df[METRIC_COLS].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(METRIC_COLS)))
ax.set_yticks(range(len(METRIC_COLS)))
ax.set_xticklabels(METRIC_COLS, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(METRIC_COLS, fontsize=8)
for i in range(len(METRIC_COLS)):
    for j in range(len(METRIC_COLS)):
        v = corr.iloc[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if abs(v) > 0.5 else 'black')
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Metric correlation matrix')
plt.tight_layout()
plt.show()

## Composite quality score

Each metric is rank-normalised to [0, 1] (higher = better quality), then combined
into a weighted average. The weights below are a starting point; adjust them after
seeing how individual metrics correlate with visual quality.

In [ ]:
# Direction: for each metric, does a HIGHER raw value mean better or worse quality?
# 'lower_is_better' metrics get inverted after ranking.
METRIC_DIRECTION = {
    'zero_frac':          'lower_is_better',
    'dynamic_range':      'higher_is_better',
    'water_col_mean':     'lower_is_better',
    'water_col_std':      'lower_is_better',
    'bottom_coverage':    'higher_is_better',
    'bottom_smoothness':  'lower_is_better',
    'bottom_frac':        'neutral',          # extreme in either direction is bad
    'bottom_depth_range': 'lower_is_better',
    'bottom_gradient_max':'lower_is_better',
    'boundary_contrast':  'higher_is_better',
    'freq_coherence':     'higher_is_better',
}

METRIC_WEIGHTS = {
    'zero_frac':          1.0,
    'dynamic_range':      1.0,
    'water_col_mean':     0.5,
    'water_col_std':      0.5,
    'bottom_coverage':    2.0,
    'bottom_smoothness':  1.5,
    'bottom_frac':        0.5,
    'bottom_depth_range': 1.0,
    'bottom_gradient_max':1.0,
    'boundary_contrast':  2.0,
    'freq_coherence':     1.0,
}

def rank_normalise(series, direction):
    """Rank-normalise a series to [0, 1] where 1 = best quality."""
    ranked = series.rank(pct=True, na_option='bottom')
    if direction == 'lower_is_better':
        ranked = 1.0 - ranked
    elif direction == 'neutral':
        # Penalise distance from the median
        median = series.median()
        dist = (series - median).abs()
        ranked = 1.0 - dist.rank(pct=True, na_option='bottom')
    return ranked

# Compute rank-normalised columns and composite score
score = np.zeros(len(df))
total_weight = 0

for col in METRIC_COLS:
    direction = METRIC_DIRECTION[col]
    weight = METRIC_WEIGHTS[col]
    rn = rank_normalise(df[col], direction)
    df[f'{col}_rn'] = rn
    score += weight * rn.fillna(0).values
    total_weight += weight

df['quality_score'] = score / total_weight

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df['quality_score'], bins=60, color='steelblue', edgecolor='none', alpha=0.8)
ax.set_xlabel('Composite quality score')
ax.set_ylabel('Count')
ax.set_title('Distribution of composite quality scores (higher = better)')
ax.axvline(df['quality_score'].quantile(0.05), color='red', ls='--', label='p5 (worst 5%)')
ax.axvline(df['quality_score'].quantile(0.95), color='green', ls='--', label='p95 (best 5%)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Quality score range: [{df["quality_score"].min():.3f}, {df["quality_score"].max():.3f}]')
print(f'Worst 5% threshold: {df["quality_score"].quantile(0.05):.3f}')
print(f'Best 5% threshold:  {df["quality_score"].quantile(0.95):.3f}')

## Visualization helpers

In [ ]:
def load_patch_by_uid(uid, patch_dir=PATCH_DIR):
    """Load a single patch + mask from disk given its uid.

    Parameters
    ----------
    uid : str, format '{npz_stem}::{patch_index}'

    Returns
    -------
    patch : ndarray (3, H, W)
    mask  : ndarray (H, W)
    """
    stem, idx_str = uid.split('::')
    idx = int(idx_str)
    data = np.load(patch_dir / f'{stem}.npz')
    return data['patches'][idx], data['masks'][idx]


def plot_patch_with_mask(patch, mask, uid, ax, zoom_bottom=True, margin=150,
                         subtitle=None):
    """Show ch0 echogram with bottom line overlay on a given axes.

    When zoom_bottom is True, the view is cropped to +-margin samples around
    the median bottom depth so the boundary region is visible.
    subtitle is shown below the plot as an xlabel (e.g. source file path).
    """
    ch0 = patch[0]
    bottom = get_bottom_line(mask)
    valid = bottom >= 0
    H, W = ch0.shape

    if zoom_bottom and valid.any():
        med = int(np.median(bottom[valid]))
        row_lo = max(0, med - margin)
        row_hi = min(H, med + margin)
        ch0_crop = ch0[row_lo:row_hi, :]
        bottom_shifted = np.where(valid, bottom - row_lo, -1)
    else:
        ch0_crop = ch0
        bottom_shifted = bottom
        row_lo = 0

    ax.imshow(ch0_crop, aspect='auto', cmap='viridis', vmin=0, vmax=1,
              interpolation='nearest')

    # Overlay bottom line
    xs = np.arange(W)
    valid_shifted = bottom_shifted >= 0
    if valid_shifted.any():
        ax.plot(xs[valid_shifted], bottom_shifted[valid_shifted],
                'r-', lw=1, alpha=0.9)

    ax.set_title(uid, fontsize=7, fontfamily='monospace', pad=2)
    ax.set_xticks([])
    ax.set_yticks([])
    if subtitle:
        ax.set_xlabel(subtitle, fontsize=6, fontfamily='monospace',
                      color='0.4', labelpad=2)


def _get_subtitle(uid):
    """Look up source_zarr for a uid from the dataframe."""
    row = df.loc[df['uid'] == uid]
    if not row.empty:
        return row.iloc[0]['source_zarr']
    return None


def gallery(uids, title, ncols=4, zoom_bottom=True, margin=150):
    """Plot a grid of patches identified by their uids."""
    n = len(uids)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
    if nrows == 1:
        axes = np.atleast_2d(axes)
    axes = axes.flat

    for i, uid in enumerate(uids):
        patch, mask = load_patch_by_uid(uid)
        plot_patch_with_mask(patch, mask, uid, axes[i],
                             zoom_bottom=zoom_bottom, margin=margin,
                             subtitle=_get_subtitle(uid))

    for j in range(n, len(list(axes))):
        axes[j].set_visible(False)

    fig.suptitle(title, fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


# Quick test
test_uid = df.iloc[0]['uid']
patch, mask = load_patch_by_uid(test_uid)
fig, ax = plt.subplots(figsize=(4, 3))
plot_patch_with_mask(patch, mask, test_uid, ax, subtitle=_get_subtitle(test_uid))
plt.tight_layout()
plt.show()
print(f'Test load OK: {test_uid}, patch shape={patch.shape}')

## Worst patches by individual metrics

For each metric, the 8 patches with the worst value. The red line is the
detected bottom from the mask. Check whether the metric is actually capturing
bad data — if the "worst" patches look fine, the metric is not useful.

In [ ]:
N_SHOW = 8

for col in METRIC_COLS:
    direction = METRIC_DIRECTION[col]
    if direction == 'lower_is_better':
        worst = df.nlargest(N_SHOW, col)
    elif direction == 'higher_is_better':
        worst = df.nsmallest(N_SHOW, col)
    else:  # neutral — most extreme distance from median
        median = df[col].median()
        worst = df.iloc[(df[col] - median).abs().nlargest(N_SHOW).index]

    uids = worst['uid'].tolist()
    vals = worst[col].tolist()
    sources = worst['source_zarr'].tolist()
    labels = [f'{uid}\n{col}={v:.3f}' for uid, v in zip(uids, vals)]

    # Use gallery but with extra info in title
    n = len(uids)
    ncols = 4
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3 * nrows))
    axes = np.atleast_2d(axes).flat

    for i, (uid, label, src) in enumerate(zip(uids, labels, sources)):
        patch, mask = load_patch_by_uid(uid)
        plot_patch_with_mask(patch, mask, uid, axes[i], subtitle=src)
        axes[i].set_title(label, fontsize=7, fontfamily='monospace', pad=2)

    for j in range(n, nrows * ncols):
        axes[j].set_visible(False)

    fig.suptitle(f'WORST by {col}', fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

## Best patches by individual metrics

Same layout, showing the 8 best patches per metric as a sanity check.

In [ ]:
for col in METRIC_COLS:
    direction = METRIC_DIRECTION[col]
    if direction == 'lower_is_better':
        best = df.nsmallest(N_SHOW, col)
    elif direction == 'higher_is_better':
        best = df.nlargest(N_SHOW, col)
    else:
        median = df[col].median()
        best = df.iloc[(df[col] - median).abs().nsmallest(N_SHOW).index]

    uids = best['uid'].tolist()
    vals = best[col].tolist()
    sources = best['source_zarr'].tolist()
    labels = [f'{uid}\n{col}={v:.3f}' for uid, v in zip(uids, vals)]

    n = len(uids)
    ncols = 4
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3 * nrows))
    axes = np.atleast_2d(axes).flat

    for i, (uid, label, src) in enumerate(zip(uids, labels, sources)):
        patch, mask = load_patch_by_uid(uid)
        plot_patch_with_mask(patch, mask, uid, axes[i], subtitle=src)
        axes[i].set_title(label, fontsize=7, fontfamily='monospace', pad=2)

    for j in range(n, nrows * ncols):
        axes[j].set_visible(False)

    fig.suptitle(f'BEST by {col}', fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

## Composite score: worst and best 20

The composite score combines all metrics. If it works well, the worst-20 should
all look visually bad and the best-20 should all look clean.

In [ ]:
worst_20 = df.nsmallest(20, 'quality_score')
gallery(worst_20['uid'].tolist(), 'WORST 20 by composite quality score', ncols=5)

best_20 = df.nlargest(20, 'quality_score')
gallery(best_20['uid'].tolist(), 'BEST 20 by composite quality score', ncols=5)

## Paginated browser

Sorted by composite quality score (worst first). Change `PAGE` and re-run to
browse through the dataset. Each title is a copy-pasteable uid.

In [ ]:
PAGE = 0        # change this and re-run
PER_PAGE = 16   # 4x4 grid

df_sorted = df.sort_values('quality_score', ascending=True).reset_index(drop=True)
total_pages = int(np.ceil(len(df_sorted) / PER_PAGE))

start = PAGE * PER_PAGE
end = min(start + PER_PAGE, len(df_sorted))
page_df = df_sorted.iloc[start:end]

print(f'Page {PAGE + 1}/{total_pages}  |  patches {start}–{end - 1}  |  '
      f'quality_score range: [{page_df["quality_score"].min():.3f}, '
      f'{page_df["quality_score"].max():.3f}]')

uids = page_df['uid'].tolist()
scores = page_df['quality_score'].tolist()
sources = page_df['source_zarr'].tolist()

ncols = 4
nrows = int(np.ceil(len(uids) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows))
axes = np.atleast_2d(axes).flat

for i, (uid, score, src) in enumerate(zip(uids, scores, sources)):
    patch, mask = load_patch_by_uid(uid)
    plot_patch_with_mask(patch, mask, uid, axes[i], subtitle=src)
    axes[i].set_title(f'{uid}\nq={score:.3f}', fontsize=7,
                      fontfamily='monospace', pad=2)

for j in range(len(uids), nrows * ncols):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## Zoomed bottom view

For a selection of patches, show both the full echogram and a tight crop around
the bottom boundary. This makes it easier to judge mask alignment.

In [ ]:
# Pick a handful of patches spanning the quality range to compare
quantiles = [0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]
sample_uids = []
for q in quantiles:
    target = df['quality_score'].quantile(q)
    idx = (df['quality_score'] - target).abs().idxmin()
    sample_uids.append(df.loc[idx, 'uid'])

fig, axes = plt.subplots(len(sample_uids), 2, figsize=(12, 3 * len(sample_uids)))

for row, uid in enumerate(sample_uids):
    patch, mask = load_patch_by_uid(uid)
    q_score = df.loc[df['uid'] == uid, 'quality_score'].values[0]

    # Full view (no zoom)
    plot_patch_with_mask(patch, mask, uid, axes[row, 0], zoom_bottom=False)
    axes[row, 0].set_title(f'{uid}  q={q_score:.3f}\n(full)', fontsize=7,
                           fontfamily='monospace', pad=2)

    # Zoomed to bottom +-80 samples
    plot_patch_with_mask(patch, mask, uid, axes[row, 1], zoom_bottom=True, margin=80)
    axes[row, 1].set_title(f'{uid}  q={q_score:.3f}\n(zoomed)', fontsize=7,
                           fontfamily='monospace', pad=2)

plt.tight_layout()
plt.show()

## Per-file quality summary

Some entire files might be consistently bad. This groups patches by source zarr
and shows mean quality score per file, sorted worst-first.

In [ ]:
file_summary = (
    df.groupby('source_zarr')
    .agg(
        n_patches=('uid', 'count'),
        mean_quality=('quality_score', 'mean'),
        min_quality=('quality_score', 'min'),
        mean_zero_frac=('zero_frac', 'mean'),
        mean_bottom_coverage=('bottom_coverage', 'mean'),
        mean_boundary_contrast=('boundary_contrast', 'mean'),
        mean_bottom_smoothness=('bottom_smoothness', 'mean'),
    )
    .sort_values('mean_quality', ascending=True)
)

print(file_summary.to_string())
print(f'\nTotal unique source zarrs: {len(file_summary)}')

# Highlight files where mean quality is below the 10th percentile
threshold = file_summary['mean_quality'].quantile(0.10)
bad_files = file_summary[file_summary['mean_quality'] < threshold]
if len(bad_files) > 0:
    print(f'\nFiles with mean quality below p10 ({threshold:.3f}):')
    for name, row in bad_files.iterrows():
        print(f'  {name}: mean_q={row["mean_quality"]:.3f}, '
              f'n_patches={row["n_patches"]:.0f}, '
              f'mean_zero_frac={row["mean_zero_frac"]:.3f}')

## Export flagged patches

After browsing, paste the uids of bad patches into the list below and run the
cell. The list is saved to a JSON file that can be loaded during training to
exclude these patches.

In [ ]:
# ---- Paste bad uids here ----
flagged_uids = [
    # 'D20240131-T092044_721e57f964fb::3',
    # 'D20240131-T093642_977ff50a03aa::7',
]

# Option: auto-flag everything below a quality threshold
AUTO_FLAG_THRESHOLD = None  # set to e.g. 0.30 to auto-flag
if AUTO_FLAG_THRESHOLD is not None:
    auto_flagged = df.loc[df['quality_score'] < AUTO_FLAG_THRESHOLD, 'uid'].tolist()
    flagged_uids = list(set(flagged_uids + auto_flagged))
    print(f'Auto-flagged {len(auto_flagged)} patches below quality_score={AUTO_FLAG_THRESHOLD}')

if flagged_uids:
    with open(FLAGGED_JSON, 'w') as f:
        json.dump({'flagged_uids': sorted(flagged_uids)}, f, indent=2)
    print(f'Saved {len(flagged_uids)} flagged uids to {FLAGGED_JSON}')
else:
    print('No patches flagged yet. Paste uids into the list above and re-run.')

## Using the flagged list in training

To filter out flagged patches during training, load the JSON and skip matching
uids when building the dataset. Example integration with `EchogramPatchDataset`:

In [ ]:
# Example: how to filter flagged patches when loading data
#
# import json
# with open('Datasets/flagged_bad_patches.json') as f:
#     flagged = set(json.load(f)['flagged_uids'])
#
# # When building patches from npz files:
# for npz_path in npz_files:
#     stem = npz_path.stem
#     data = np.load(npz_path)
#     for i in range(data['patches'].shape[0]):
#         uid = f'{stem}::{i}'
#         if uid in flagged:
#             continue  # skip this patch
#         # ... use patch normally
#
# Alternatively, filter by quality_score threshold using the CSV:
# df = pd.read_csv('Datasets/patch_quality_metrics.csv')
# good_uids = set(df.loc[df['quality_score'] >= 0.30, 'uid'])

print('See code comments above for integration examples.')